# Data set file structure

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth < 3:  # show files only up to 3 levels deep
        for f in files:
            print(f"  {indent}{f}")

In [ ]:
import os

root = "/kaggle/input/datasets/ayushalia/kitpri-v2/kitpri_v2/audio_32k"

cooking    = os.listdir(f"{root}/cooking")
noncooking = os.listdir(f"{root}/noncooking")

print(f"cooking    : {len(cooking)} files")
print(f"noncooking : {len(noncooking)} files")
print(f"total      : {len(cooking) + len(noncooking)} files")

print(f"\nCooking path    : {root}/cooking")
print(f"Noncooking path : {root}/noncooking")

print("\nSample cooking files:")
for f in sorted(cooking)[:5]:
    print(f"  {f}")

print("\nSample noncooking files:")
for f in sorted(noncooking)[:5]:
    print(f"  {f}")

# **#from here on new data set**> 

In [ ]:
!pip install -q timm panns-inference

In [ ]:
# =============================================================================
# PHASE 7 — CNN14 + AST Ensemble Training
# Samsung PRISM — Kitchen Audio Classification (kitpri_v2)
# Target: F1 > 0.95 | RAM < 60 MB
# =============================================================================
# CELL 1 (separate cell): !pip install -q timm panns-inference
# CELL 2: Paste everything below into ONE cell and run
# =============================================================================

import os, sys, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

import librosa
import timm
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, classification_report

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    DATA_ROOT    = "/kaggle/input/datasets/ayushalia/kitpri-v2/kitpri_v2"
    AUDIO_DIR    = f"{DATA_ROOT}/audio_32k"
    TRAIN_CSV    = f"{DATA_ROOT}/metadata/train.csv"
    VAL_CSV      = f"{DATA_ROOT}/metadata/val.csv"
    TEST_CSV     = f"{DATA_ROOT}/metadata/test.csv"
    OUTPUT_DIR   = "/kaggle/working/phase7_outputs"

    SR           = 32000
    DURATION     = 10
    N_SAMPLES    = SR * DURATION
    N_FFT        = 1024
    HOP_LENGTH   = 320
    N_MELS       = 64
    F_MIN        = 50
    F_MAX        = 14000

    EPOCHS       = 50          # was 30
    BATCH_SIZE   = 16
    LR           = 1e-4        # was 3e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS  = 2
    SEED         = 42

    CNN14_WEIGHT = 0.45
    AST_WEIGHT   = 0.55
    PATIENCE     = 15          # was 7

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print(f"Device : {CFG.DEVICE}")
print(f"Root   : {CFG.DATA_ROOT}")


# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(seed=CFG.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()


# ── 3. Filter CSVs to existing files only ─────────────────────────────────────
print("\nFiltering CSVs to existing files only...")
for attr, csv_path in [("TRAIN_CSV", CFG.TRAIN_CSV),
                        ("VAL_CSV",   CFG.VAL_CSV),
                        ("TEST_CSV",  CFG.TEST_CSV)]:
    df     = pd.read_csv(csv_path)
    before = len(df)
    df     = df[df["file_path"].apply(
                    lambda p: os.path.exists(f"{CFG.DATA_ROOT}/{p}")
                )].reset_index(drop=True)
    after  = len(df)
    out    = f"/kaggle/working/{os.path.basename(csv_path)}"
    df.to_csv(out, index=False)
    setattr(CFG, attr, out)
    print(f"  {os.path.basename(csv_path)}: {before} -> {after} rows "
          f"(cooking={( df['label']==1).sum()}, noncooking={(df['label']==0).sum()})")

print("CSV filtering done.\n")


# ── 4. Dataset ────────────────────────────────────────────────────────────────
class KitpriDataset(Dataset):
    def __init__(self, csv_path, augment=False):
        self.df       = pd.read_csv(csv_path)
        self.root     = Path(CFG.DATA_ROOT)
        self.augment  = augment
        self.n_samples = CFG.N_SAMPLES
        print(f"  Loaded {len(self.df)} clips from {Path(csv_path).name} | "
              f"cooking={(self.df['label']==1).sum()} | "
              f"noncooking={(self.df['label']==0).sum()}")

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        wav, _ = librosa.load(str(path), sr=CFG.SR, mono=True, duration=CFG.DURATION)
        if len(wav) < self.n_samples:
            wav = np.pad(wav, (0, self.n_samples - len(wav)))
        else:
            wav = wav[:self.n_samples]
        return wav.astype(np.float32)

    def _to_melspec(self, wav):
        mel = librosa.feature.melspectrogram(
            y=wav, sr=CFG.SR,
            n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, fmin=CFG.F_MIN, fmax=CFG.F_MAX
        )
        mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db

    def _augment(self, wav):
        gain  = np.random.uniform(0.7, 1.3)
        wav   = wav * gain
        shift = np.random.randint(-CFG.SR // 2, CFG.SR // 2)
        wav   = np.roll(wav, shift)
        return np.clip(wav, -1.0, 1.0)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        full_path = self.root / row["file_path"]
        wav       = self._load_audio(full_path)
        if self.augment:
            wav = self._augment(wav)
        mel   = torch.tensor(self._to_melspec(wav)).unsqueeze(0)
        label = int(row["label"])
        return mel, label, str(full_path)


def get_loaders():
    train_ds = KitpriDataset(CFG.TRAIN_CSV, augment=True)
    val_ds   = KitpriDataset(CFG.VAL_CSV,   augment=False)
    test_ds  = KitpriDataset(CFG.TEST_CSV,  augment=False)

    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CFG.BATCH_SIZE, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader


# ── 5. CNN14 Model ────────────────────────────────────────────────────────────
class CNN14(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            )

        self.bn0    = nn.BatchNorm2d(1)
        self.block1 = conv_block(1,    64)
        self.block2 = conv_block(64,   128)
        self.block3 = conv_block(128,  256)
        self.block4 = conv_block(256,  512)
        self.block5 = conv_block(512,  1024)
        self.block6 = conv_block(1024, 2048)
        self.pool   = nn.AvgPool2d(2, 2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.fc1    = nn.Linear(2048, 2048)
        self.drop   = nn.Dropout(0.5)
        self.fc_out = nn.Linear(2048, 1)
        self._init_weights()
        if pretrained:
            self._load_panns_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def _load_panns_weights(self):
        try:
            from panns_inference import AudioTagging
            at = AudioTagging(checkpoint_path=None, device="cpu")
            panns_sd = at.model.state_dict()
            our_sd = self.state_dict()
            matched, skipped = 0, 0
            for k, v in panns_sd.items():
                if k in our_sd and our_sd[k].shape == v.shape:
                    our_sd[k] = v; matched += 1
                else:
                    skipped += 1
            self.load_state_dict(our_sd, strict=False)
            print(f"  [CNN14] PANNs weights: {matched} matched, {skipped} skipped")
        except Exception as e:
            print(f"  [CNN14] PANNs unavailable ({e}), training from scratch")

    def forward(self, x):
        x = self.bn0(x)
        x = self.pool(self.block1(x))
        x = self.pool(self.block2(x))
        x = self.pool(self.block3(x))
        x = self.pool(self.block4(x))
        x = self.pool(self.block5(x))
        x = self.block6(x)
        x = self.gap(x).flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc_out(x)


# ── 6. AST Model ──────────────────────────────────────────────────────────────
class ASTModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "deit_small_patch16_224",
            pretrained=pretrained, num_classes=0, global_pool="token"
        )
        embed_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
        print(f"  [AST] DeiT-Small | embed_dim={embed_dim}")

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        return self.head(self.backbone(x))


# ── 7. Ensemble ───────────────────────────────────────────────────────────────
class CNN14_AST_Ensemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn14   = CNN14(pretrained=True)
        self.ast     = ASTModel(pretrained=True)

    def forward(self, x):
        return CFG.CNN14_WEIGHT * self.cnn14(x) + CFG.AST_WEIGHT * self.ast(x)


# ── 8. Training utilities ─────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train()
    losses, preds, targets = [], [], []
    for batch_idx, (mel, label, _) in enumerate(loader):
        mel   = mel.to(CFG.DEVICE)
        label = label.float().to(CFG.DEVICE)
        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            logit = model(mel).squeeze(1)
            loss  = criterion(logit, label)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        losses.append(loss.item())
        preds.extend(torch.sigmoid(logit).detach().cpu().numpy())
        targets.extend(label.cpu().numpy())
        if batch_idx % 50 == 0:
            print(f"    batch {batch_idx}/{len(loader)} | loss={loss.item():.4f}")
    pred_bin = (np.array(preds) >= 0.5).astype(int)
    return np.mean(losses), f1_score(targets, pred_bin), roc_auc_score(targets, preds)


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    losses, preds, targets = [], [], []
    for mel, label, _ in loader:
        mel   = mel.to(CFG.DEVICE)
        label = label.float().to(CFG.DEVICE)
        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            logit = model(mel).squeeze(1)
            loss  = criterion(logit, label)
        losses.append(loss.item())
        preds.extend(torch.sigmoid(logit).cpu().numpy())
        targets.extend(label.cpu().numpy())
    preds    = np.array(preds)
    targets  = np.array(targets)
    pred_bin = (preds >= 0.5).astype(int)
    return (np.mean(losses),
            f1_score(targets, pred_bin),
            roc_auc_score(targets, preds),
            accuracy_score(targets, pred_bin),
            preds, targets)


def train(model, train_loader, val_loader):
    pos_weight = torch.tensor([1.2]).to(CFG.DEVICE)  # upweights noncooking errors
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = OneCycleLR(optimizer, max_lr=CFG.LR,
                           steps_per_epoch=len(train_loader),
                           epochs=CFG.EPOCHS, pct_start=0.1, anneal_strategy="cos")
    scaler    = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))

    best_f1, patience_c, history = 0.0, 0, []

    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        print(f"\n── Epoch {epoch}/{CFG.EPOCHS} ──────────────────────────")
        tr_loss, tr_f1, tr_auc = train_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        va_loss, va_f1, va_auc, va_acc, _, _ = eval_epoch(model, val_loader, criterion)
        print(f"  TRAIN -> loss={tr_loss:.4f} | F1={tr_f1:.4f} | AUC={tr_auc:.4f}")
        print(f"  VAL   -> loss={va_loss:.4f} | F1={va_f1:.4f} | AUC={va_auc:.4f} | Acc={va_acc:.4f} | {time.time()-t0:.0f}s")
        history.append({"epoch": epoch, "tr_loss": tr_loss, "tr_f1": tr_f1,
                         "va_loss": va_loss, "va_f1": va_f1, "va_auc": va_auc})
        if va_f1 > best_f1:
            best_f1, patience_c = va_f1, 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_f1": va_f1, "val_auc": va_auc, "val_acc": va_acc},
                       f"{CFG.OUTPUT_DIR}/best_ensemble.pt")
            print(f"  * New best F1={best_f1:.4f} saved.")
        else:
            patience_c += 1
            print(f"  No improvement ({patience_c}/{CFG.PATIENCE})")
            if patience_c >= CFG.PATIENCE:
                print("  Early stopping.")
                break

    pd.DataFrame(history).to_csv(f"{CFG.OUTPUT_DIR}/training_history.csv", index=False)
    print(f"\nTraining complete. Best Val F1 = {best_f1:.4f}")
    return best_f1


# ── 9. Test Evaluation ────────────────────────────────────────────────────────
def evaluate_test(model, test_loader):
    ckpt_path = f"{CFG.OUTPUT_DIR}/best_ensemble.pt"
    ckpt = torch.load(ckpt_path, map_location=CFG.DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"\n-- Test Evaluation (best epoch={ckpt['epoch']}, val F1={ckpt['val_f1']:.4f}) --")
    criterion = nn.BCEWithLogitsLoss()
    _, f1, auc, acc, preds, targets = eval_epoch(model, test_loader, criterion)
    print(f"  F1={f1:.4f} | AUC={auc:.4f} | Acc={acc:.4f}")
    print(classification_report(targets, (preds >= 0.5).astype(int),
                                 target_names=["noncooking", "cooking"]))
    pd.DataFrame({"prob_cooking": preds,
                  "pred_label":   (preds >= 0.5).astype(int),
                  "true_label":   targets.astype(int)
                  }).to_csv(f"{CFG.OUTPUT_DIR}/test_predictions.csv", index=False)
    return {"f1": f1, "auc": auc, "accuracy": acc}


# ── 10. Run ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("Phase 7 -- CNN14 + AST Ensemble Training")
print("Samsung PRISM | kitpri_v2")
print("=" * 60)

print("Loading datasets...")
train_loader, val_loader, test_loader = get_loaders()

print("\nBuilding CNN14 + AST Ensemble...")
model = CNN14_AST_Ensemble().to(CFG.DEVICE)
print(f"  Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")

best_f1      = train(model, train_loader, val_loader)
test_results = evaluate_test(model, test_loader)

print("\n" + "=" * 60)
print("Phase 7 COMPLETE")
print(f"  Best Val F1  : {best_f1:.4f}")
print(f"  Test F1      : {test_results['f1']:.4f}")
print(f"  Test AUC     : {test_results['auc']:.4f}")
print(f"  Test Accuracy: {test_results['accuracy']:.4f}")
print(f"  F1 > 0.95?   : {'TARGET MET' if test_results['f1'] >= 0.95 else 'Below target'}")
print("=" * 60)

In [ ]:
import torch, numpy as np
from torch import nn
from sklearn.metrics import classification_report, f1_score

# Fix: weights_only=False
ckpt = torch.load(f"{CFG.OUTPUT_DIR}/best_ensemble.pt", 
                  map_location=CFG.DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded epoch {ckpt['epoch']} | val F1={ckpt['val_f1']:.4f}")

# Get all predictions
criterion = nn.BCEWithLogitsLoss()
_, f1, auc, acc, preds, targets = eval_epoch(model, test_loader, criterion)

print(f"\nDefault threshold (0.5):")
print(f"  F1={f1:.4f} | AUC={auc:.4f} | Acc={acc:.4f}")

# Try different thresholds to find best F1
print("\nThreshold search:")
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    pred_bin = (preds >= thresh).astype(int)
    f1_t = f1_score(targets, pred_bin)
    print(f"  thresh={thresh} -> F1={f1_t:.4f}")

In [ ]:
# model was mostly over fitting, so we made a few changes

In [ ]:
# =============================================================================
# PHASE 7 v3 — CNN14 + AST Ensemble (Anti-Overfit)
# Samsung PRISM — Kitchen Audio Classification (kitpri_v2)
# Fixes: frozen backbones, label smoothing, strong weight decay, pos_weight
# Target: F1 > 0.95
# =============================================================================
# CELL 1 (separate cell): !pip install -q timm panns-inference
# CELL 2: Paste everything below into ONE cell and run
# =============================================================================

import os, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

import librosa
import timm
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, classification_report

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True


# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    DATA_ROOT    = "/kaggle/input/datasets/ayushalia/kitpri-v2/kitpri_v2"
    AUDIO_DIR    = f"{DATA_ROOT}/audio_32k"
    TRAIN_CSV    = f"{DATA_ROOT}/metadata/train.csv"
    VAL_CSV      = f"{DATA_ROOT}/metadata/val.csv"
    TEST_CSV     = f"{DATA_ROOT}/metadata/test.csv"
    OUTPUT_DIR   = "/kaggle/working/phase7_outputs"

    SR           = 32000
    DURATION     = 10
    N_SAMPLES    = SR * DURATION
    N_FFT        = 1024
    HOP_LENGTH   = 320
    N_MELS       = 64
    F_MIN        = 50
    F_MAX        = 14000

    EPOCHS       = 50
    BATCH_SIZE   = 32          # larger = less overfit
    LR           = 5e-5        # slow learning
    WEIGHT_DECAY = 1e-2        # strong L2 regularization
    NUM_WORKERS  = 2
    SEED         = 42

    CNN14_WEIGHT = 0.45
    AST_WEIGHT   = 0.55
    PATIENCE     = 15
    LABEL_SMOOTH = 0.1         # prevents loss → 0.0000

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

# Clear old checkpoint
ckpt = f"{CFG.OUTPUT_DIR}/best_ensemble.pt"
if os.path.exists(ckpt):
    os.remove(ckpt)
    print("Old checkpoint cleared.")

print(f"Device : {CFG.DEVICE}")
print(f"Root   : {CFG.DATA_ROOT}")


# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(seed=CFG.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()


# ── 3. Filter CSVs to existing files only ────────────────────────────────────
print("\nFiltering CSVs to existing files only...")
for attr, csv_path in [("TRAIN_CSV", CFG.TRAIN_CSV),
                        ("VAL_CSV",   CFG.VAL_CSV),
                        ("TEST_CSV",  CFG.TEST_CSV)]:
    df     = pd.read_csv(csv_path)
    before = len(df)
    df     = df[df["file_path"].apply(
                    lambda p: os.path.exists(f"{CFG.DATA_ROOT}/{p}")
                )].reset_index(drop=True)
    after  = len(df)
    out    = f"/kaggle/working/{os.path.basename(csv_path)}"
    df.to_csv(out, index=False)
    setattr(CFG, attr, out)
    print(f"  {os.path.basename(csv_path)}: {before} -> {after} rows "
          f"(cooking={(df['label']==1).sum()}, noncooking={(df['label']==0).sum()})")
print("Done.\n")


# ── 4. Dataset ────────────────────────────────────────────────────────────────
class KitpriDataset(Dataset):
    def __init__(self, csv_path, augment=False):
        self.df        = pd.read_csv(csv_path)
        self.root      = Path(CFG.DATA_ROOT)
        self.augment   = augment
        self.n_samples = CFG.N_SAMPLES
        print(f"  Loaded {len(self.df)} clips from {Path(csv_path).name} | "
              f"cooking={(self.df['label']==1).sum()} | "
              f"noncooking={(self.df['label']==0).sum()}")

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        wav, _ = librosa.load(str(path), sr=CFG.SR, mono=True, duration=CFG.DURATION)
        if len(wav) < self.n_samples:
            wav = np.pad(wav, (0, self.n_samples - len(wav)))
        else:
            wav = wav[:self.n_samples]
        return wav.astype(np.float32)

    def _to_melspec(self, wav):
        mel = librosa.feature.melspectrogram(
            y=wav, sr=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, fmin=CFG.F_MIN, fmax=CFG.F_MAX
        )
        mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db

    def _augment(self, wav):
        # Random gain
        wav = wav * np.random.uniform(0.7, 1.3)
        # Random time shift
        wav = np.roll(wav, np.random.randint(-CFG.SR // 2, CFG.SR // 2))
        # Random noise
        wav = wav + np.random.randn(len(wav)).astype(np.float32) * 0.005
        return np.clip(wav, -1.0, 1.0)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        full_path = self.root / row["file_path"]
        wav       = self._load_audio(full_path)
        if self.augment:
            wav = self._augment(wav)
        mel   = torch.tensor(self._to_melspec(wav)).unsqueeze(0)
        label = int(row["label"])
        return mel, label, str(full_path)


def get_loaders():
    train_ds = KitpriDataset(CFG.TRAIN_CSV, augment=True)
    val_ds   = KitpriDataset(CFG.VAL_CSV,   augment=False)
    test_ds  = KitpriDataset(CFG.TEST_CSV,  augment=False)
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CFG.BATCH_SIZE, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader


# ── 5. CNN14 Model (with frozen backbone option) ──────────────────────────────
class CNN14(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            )

        self.bn0    = nn.BatchNorm2d(1)
        self.block1 = conv_block(1,    64)
        self.block2 = conv_block(64,   128)
        self.block3 = conv_block(128,  256)
        self.block4 = conv_block(256,  512)
        self.block5 = conv_block(512,  1024)
        self.block6 = conv_block(1024, 2048)
        self.pool   = nn.AvgPool2d(2, 2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.fc1    = nn.Linear(2048, 512)   # smaller head: 2048→512 (was 2048→2048)
        self.drop   = nn.Dropout(0.6)        # stronger dropout (was 0.5)
        self.fc_out = nn.Linear(512, 1)
        self._init_weights()

        if pretrained:
            self._load_panns_weights()

        # Freeze all conv blocks, only train fc head
        if freeze_backbone:
            for name, param in self.named_parameters():
                if "fc1" not in name and "fc_out" not in name:
                    param.requires_grad = False
            print("  [CNN14] Backbone frozen — training classifier head only")

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def _load_panns_weights(self):
        try:
            from panns_inference import AudioTagging
            at = AudioTagging(checkpoint_path=None, device="cpu")
            panns_sd = at.model.state_dict()
            our_sd = self.state_dict()
            matched, skipped = 0, 0
            for k, v in panns_sd.items():
                if k in our_sd and our_sd[k].shape == v.shape:
                    our_sd[k] = v; matched += 1
                else:
                    skipped += 1
            self.load_state_dict(our_sd, strict=False)
            print(f"  [CNN14] PANNs weights: {matched} matched, {skipped} skipped")
        except Exception as e:
            print(f"  [CNN14] PANNs unavailable ({e}), training from scratch")

    def forward(self, x):
        x = self.bn0(x)
        x = self.pool(self.block1(x))
        x = self.pool(self.block2(x))
        x = self.pool(self.block3(x))
        x = self.pool(self.block4(x))
        x = self.pool(self.block5(x))
        x = self.block6(x)
        x = self.gap(x).flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc_out(x)


# ── 6. AST Model (with frozen backbone option) ────────────────────────────────
class ASTModel(nn.Module):
    def __init__(self, pretrained=True, freeze_backbone=True):
        super().__init__()
        self.backbone = timm.create_model(
            "deit_small_patch16_224",
            pretrained=pretrained, num_classes=0, global_pool="token"
        )
        embed_dim = self.backbone.num_features   # 384

        # Freeze all but last 2 transformer blocks
        if freeze_backbone:
            for name, param in self.backbone.named_parameters():
                param.requires_grad = False
            # Unfreeze last 2 blocks only
            for block in self.backbone.blocks[-2:]:
                for param in block.parameters():
                    param.requires_grad = True
            print(f"  [AST] Backbone mostly frozen — last 2 blocks trainable")

        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),   # smaller head (was 256)
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )
        print(f"  [AST] DeiT-Small | embed_dim={embed_dim}")

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        return self.head(self.backbone(x))


# ── 7. Ensemble ───────────────────────────────────────────────────────────────
class CNN14_AST_Ensemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn14 = CNN14(pretrained=True,  freeze_backbone=True)
        self.ast   = ASTModel(pretrained=True, freeze_backbone=True)

    def forward(self, x):
        return CFG.CNN14_WEIGHT * self.cnn14(x) + CFG.AST_WEIGHT * self.ast(x)


# ── 8. Label smoothing helper ─────────────────────────────────────────────────
def smooth_labels(targets, smoothing=CFG.LABEL_SMOOTH):
    """Prevent overconfidence: 0 → 0.05, 1 → 0.95"""
    return targets * (1 - smoothing) + 0.5 * smoothing


# ── 9. Training utilities ─────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train()
    losses, preds, targets = [], [], []
    for batch_idx, (mel, label, _) in enumerate(loader):
        mel   = mel.to(CFG.DEVICE)
        label = label.float().to(CFG.DEVICE)
        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            logit = model(mel).squeeze(1)
            loss  = criterion(logit, smooth_labels(label))   # label smoothing
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        losses.append(loss.item())
        preds.extend(torch.sigmoid(logit).detach().cpu().numpy())
        targets.extend(label.cpu().numpy())
        if batch_idx % 30 == 0:
            print(f"    batch {batch_idx}/{len(loader)} | loss={loss.item():.4f}")
    pred_bin = (np.array(preds) >= 0.5).astype(int)
    return np.mean(losses), f1_score(targets, pred_bin), roc_auc_score(targets, preds)


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    losses, preds, targets = [], [], []
    for mel, label, _ in loader:
        mel   = mel.to(CFG.DEVICE)
        label = label.float().to(CFG.DEVICE)
        with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
            logit = model(mel).squeeze(1)
            loss  = criterion(logit, label)
        losses.append(loss.item())
        preds.extend(torch.sigmoid(logit).cpu().numpy())
        targets.extend(label.cpu().numpy())
    preds    = np.array(preds)
    targets  = np.array(targets)
    pred_bin = (preds >= 0.5).astype(int)
    return (np.mean(losses),
            f1_score(targets, pred_bin),
            roc_auc_score(targets, preds),
            accuracy_score(targets, pred_bin),
            preds, targets)


def train(model, train_loader, val_loader):
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([1.2]).to(CFG.DEVICE)  # fix noncooking bias
    )
    # Only optimize trainable parameters
    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"\n  Trainable parameters: {sum(p.numel() for p in trainable)/1e6:.1f} M "
          f"(frozen: {sum(p.numel() for p in model.parameters() if not p.requires_grad)/1e6:.1f} M)")

    optimizer = AdamW(trainable, lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = OneCycleLR(optimizer, max_lr=CFG.LR,
                           steps_per_epoch=len(train_loader),
                           epochs=CFG.EPOCHS, pct_start=0.1, anneal_strategy="cos")
    scaler    = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))

    best_f1, patience_c, history = 0.0, 0, []

    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        print(f"\n── Epoch {epoch}/{CFG.EPOCHS} ──────────────────────────")
        tr_loss, tr_f1, tr_auc = train_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        va_loss, va_f1, va_auc, va_acc, _, _ = eval_epoch(model, val_loader, criterion)
        print(f"  TRAIN -> loss={tr_loss:.4f} | F1={tr_f1:.4f} | AUC={tr_auc:.4f}")
        print(f"  VAL   -> loss={va_loss:.4f} | F1={va_f1:.4f} | AUC={va_auc:.4f} | Acc={va_acc:.4f} | {time.time()-t0:.0f}s")
        history.append({"epoch": epoch, "tr_loss": tr_loss, "tr_f1": tr_f1,
                         "va_loss": va_loss, "va_f1": va_f1, "va_auc": va_auc})
        if va_f1 > best_f1:
            best_f1, patience_c = va_f1, 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_f1": va_f1, "val_auc": va_auc, "val_acc": va_acc},
                       f"{CFG.OUTPUT_DIR}/best_ensemble.pt")
            print(f"  * New best F1={best_f1:.4f} saved.")
        else:
            patience_c += 1
            print(f"  No improvement ({patience_c}/{CFG.PATIENCE})")
            if patience_c >= CFG.PATIENCE:
                print("  Early stopping.")
                break

    pd.DataFrame(history).to_csv(f"{CFG.OUTPUT_DIR}/training_history.csv", index=False)
    print(f"\nTraining complete. Best Val F1 = {best_f1:.4f}")
    return best_f1


# ── 10. Test Evaluation ───────────────────────────────────────────────────────
def evaluate_test(model, test_loader):
    ckpt = torch.load(f"{CFG.OUTPUT_DIR}/best_ensemble.pt",
                      map_location=CFG.DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    print(f"\n-- Test Evaluation (best epoch={ckpt['epoch']}, val F1={ckpt['val_f1']:.4f}) --")
    criterion = nn.BCEWithLogitsLoss()
    _, f1, auc, acc, preds, targets = eval_epoch(model, test_loader, criterion)

    # Threshold search
    print("\nThreshold search:")
    best_thresh, best_f1_t = 0.5, 0.0
    for thresh in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
        f1_t = f1_score(targets, (preds >= thresh).astype(int))
        marker = " <-- best" if f1_t > best_f1_t else ""
        print(f"  thresh={thresh} -> F1={f1_t:.4f}{marker}")
        if f1_t > best_f1_t:
            best_f1_t, best_thresh = f1_t, thresh

    print(f"\nFinal results at threshold={best_thresh}:")
    pred_bin = (preds >= best_thresh).astype(int)
    print(f"  F1={best_f1_t:.4f} | AUC={auc:.4f} | Acc={accuracy_score(targets, pred_bin):.4f}")
    print(classification_report(targets, pred_bin, target_names=["noncooking", "cooking"]))

    pd.DataFrame({"prob_cooking": preds, "pred_label": pred_bin,
                  "true_label": targets.astype(int)
                  }).to_csv(f"{CFG.OUTPUT_DIR}/test_predictions.csv", index=False)
    return {"f1": best_f1_t, "auc": auc, "accuracy": accuracy_score(targets, pred_bin),
            "threshold": best_thresh}


# ── 11. Run ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("Phase 7 v3 -- CNN14 + AST Ensemble (Anti-Overfit)")
print("Samsung PRISM | kitpri_v2")
print("=" * 60)

print("Loading datasets...")
train_loader, val_loader, test_loader = get_loaders()

print("\nBuilding CNN14 + AST Ensemble...")
model = CNN14_AST_Ensemble().to(CFG.DEVICE)
total  = sum(p.numel() for p in model.parameters()) / 1e6
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad) / 1e6
print(f"  Total: {total:.1f} M | Frozen: {frozen:.1f} M | Trainable: {total-frozen:.1f} M")

best_f1      = train(model, train_loader, val_loader)
test_results = evaluate_test(model, test_loader)

print("\n" + "=" * 60)
print("Phase 7 v3 COMPLETE")
print(f"  Best Val F1  : {best_f1:.4f}")
print(f"  Test F1      : {test_results['f1']:.4f}")
print(f"  Test AUC     : {test_results['auc']:.4f}")
print(f"  Test Accuracy: {test_results['accuracy']:.4f}")
print(f"  Threshold    : {test_results['threshold']}")
print(f"  F1 > 0.95?   : {'TARGET MET' if test_results['f1'] >= 0.95 else 'Below target'}")
print("=" * 60)

In [ ]:
# Phase 7 -> v4 version for the model

In [ ]:
!pip install -q timm panns-inference

In [ ]:
# =============================================================================
# PHASE 7 v4 — CNN14 + AST Ensemble (Staged Unfreezing)
# Samsung PRISM — Kitchen Audio Classification (kitpri_v2)
# Strategy: Train head first (5 epochs), then unfreeze full model with low LR
# Target: F1 > 0.95
# =============================================================================
# CELL 1 (separate cell): !pip install -q timm panns-inference
# CELL 2: Paste everything below into ONE cell and run
# =============================================================================

import os, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR

import librosa
import timm
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, classification_report

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True


# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    DATA_ROOT    = "/kaggle/input/datasets/ayushalia/kitpri-v2/kitpri_v2"
    AUDIO_DIR    = f"{DATA_ROOT}/audio_32k"
    TRAIN_CSV    = f"{DATA_ROOT}/metadata/train.csv"
    VAL_CSV      = f"{DATA_ROOT}/metadata/val.csv"
    TEST_CSV     = f"{DATA_ROOT}/metadata/test.csv"
    OUTPUT_DIR   = "/kaggle/working/phase7_outputs"

    SR           = 32000
    DURATION     = 10
    N_SAMPLES    = SR * DURATION
    N_FFT        = 1024
    HOP_LENGTH   = 320
    N_MELS       = 64
    F_MIN        = 50
    F_MAX        = 14000

    # Stage 1: train heads only
    STAGE1_EPOCHS  = 8
    STAGE1_LR      = 3e-4

    # Stage 2: unfreeze everything, fine-tune at low LR
    STAGE2_EPOCHS  = 40
    STAGE2_LR      = 5e-5

    BATCH_SIZE   = 24
    WEIGHT_DECAY = 5e-3
    NUM_WORKERS  = 2
    SEED         = 42

    CNN14_WEIGHT = 0.45
    AST_WEIGHT   = 0.55
    PATIENCE     = 12
    LABEL_SMOOTH = 0.05        # lighter smoothing than v3

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

# Clear old checkpoint
ckpt_path = f"{CFG.OUTPUT_DIR}/best_ensemble.pt"
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print("Old checkpoint cleared.")

print(f"Device : {CFG.DEVICE}")
print(f"Root   : {CFG.DATA_ROOT}")


# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(seed=CFG.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()


# ── 3. Filter CSVs ────────────────────────────────────────────────────────────
print("\nFiltering CSVs to existing files only...")
for attr, csv_path in [("TRAIN_CSV", CFG.TRAIN_CSV),
                        ("VAL_CSV",   CFG.VAL_CSV),
                        ("TEST_CSV",  CFG.TEST_CSV)]:
    df     = pd.read_csv(csv_path)
    before = len(df)
    df     = df[df["file_path"].apply(
                    lambda p: os.path.exists(f"{CFG.DATA_ROOT}/{p}")
                )].reset_index(drop=True)
    after  = len(df)
    out    = f"/kaggle/working/{os.path.basename(csv_path)}"
    df.to_csv(out, index=False)
    setattr(CFG, attr, out)
    print(f"  {os.path.basename(csv_path)}: {before} -> {after} rows "
          f"(cooking={(df['label']==1).sum()}, noncooking={(df['label']==0).sum()})")
print("Done.\n")


# ── 4. Dataset ────────────────────────────────────────────────────────────────
class KitpriDataset(Dataset):
    def __init__(self, csv_path, augment=False):
        self.df        = pd.read_csv(csv_path)
        self.root      = Path(CFG.DATA_ROOT)
        self.augment   = augment
        self.n_samples = CFG.N_SAMPLES
        print(f"  Loaded {len(self.df)} clips | "
              f"cooking={(self.df['label']==1).sum()} | "
              f"noncooking={(self.df['label']==0).sum()}")

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        wav, _ = librosa.load(str(path), sr=CFG.SR, mono=True, duration=CFG.DURATION)
        if len(wav) < self.n_samples:
            wav = np.pad(wav, (0, self.n_samples - len(wav)))
        else:
            wav = wav[:self.n_samples]
        return wav.astype(np.float32)

    def _to_melspec(self, wav):
        mel = librosa.feature.melspectrogram(
            y=wav, sr=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, fmin=CFG.F_MIN, fmax=CFG.F_MAX
        )
        mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db

    def _augment(self, wav):
        wav = wav * np.random.uniform(0.7, 1.3)
        wav = np.roll(wav, np.random.randint(-CFG.SR // 2, CFG.SR // 2))
        wav = wav + np.random.randn(len(wav)).astype(np.float32) * 0.005
        return np.clip(wav, -1.0, 1.0)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        full_path = self.root / row["file_path"]
        wav       = self._load_audio(full_path)
        if self.augment:
            wav = self._augment(wav)
        mel   = torch.tensor(self._to_melspec(wav)).unsqueeze(0)
        label = int(row["label"])
        return mel, label, str(full_path)


def get_loaders():
    train_ds = KitpriDataset(CFG.TRAIN_CSV, augment=True)
    val_ds   = KitpriDataset(CFG.VAL_CSV,   augment=False)
    test_ds  = KitpriDataset(CFG.TEST_CSV,  augment=False)
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CFG.BATCH_SIZE, shuffle=False,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader


# ── 5. CNN14 ──────────────────────────────────────────────────────────────────
class CNN14(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            )

        self.bn0    = nn.BatchNorm2d(1)
        self.block1 = conv_block(1,    64)
        self.block2 = conv_block(64,   128)
        self.block3 = conv_block(128,  256)
        self.block4 = conv_block(256,  512)
        self.block5 = conv_block(512,  1024)
        self.block6 = conv_block(1024, 2048)
        self.pool   = nn.AvgPool2d(2, 2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.fc1    = nn.Linear(2048, 512)
        self.drop   = nn.Dropout(0.5)
        self.fc_out = nn.Linear(512, 1)
        self._init_weights()
        if pretrained:
            self._load_panns_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def _load_panns_weights(self):
        try:
            from panns_inference import AudioTagging
            at = AudioTagging(checkpoint_path=None, device="cpu")
            panns_sd = at.model.state_dict()
            our_sd = self.state_dict()
            matched, skipped = 0, 0
            for k, v in panns_sd.items():
                if k in our_sd and our_sd[k].shape == v.shape:
                    our_sd[k] = v; matched += 1
                else:
                    skipped += 1
            self.load_state_dict(our_sd, strict=False)
            print(f"  [CNN14] PANNs weights: {matched} matched, {skipped} skipped")
        except Exception as e:
            print(f"  [CNN14] PANNs unavailable, training from scratch")

    def freeze_backbone(self):
        for name, param in self.named_parameters():
            if "fc1" not in name and "fc_out" not in name:
                param.requires_grad = False

    def unfreeze_all(self):
        for param in self.parameters():
            param.requires_grad = True

    def forward(self, x):
        x = self.bn0(x)
        x = self.pool(self.block1(x))
        x = self.pool(self.block2(x))
        x = self.pool(self.block3(x))
        x = self.pool(self.block4(x))
        x = self.pool(self.block5(x))
        x = self.block6(x)
        x = self.gap(x).flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc_out(x)


# ── 6. AST ────────────────────────────────────────────────────────────────────
class ASTModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "deit_small_patch16_224",
            pretrained=pretrained, num_classes=0, global_pool="token"
        )
        embed_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1)
        )
        print(f"  [AST] DeiT-Small | embed_dim={embed_dim}")

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_last_n_blocks(self, n=4):
        for param in self.backbone.parameters():
            param.requires_grad = False
        for block in self.backbone.blocks[-n:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in self.backbone.norm.parameters():
            param.requires_grad = True

    def unfreeze_all(self):
        for param in self.parameters():
            param.requires_grad = True

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        return self.head(self.backbone(x))


# ── 7. Ensemble ───────────────────────────────────────────────────────────────
class CNN14_AST_Ensemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn14 = CNN14(pretrained=True)
        self.ast   = ASTModel(pretrained=True)

    def forward(self, x):
        return CFG.CNN14_WEIGHT * self.cnn14(x) + CFG.AST_WEIGHT * self.ast(x)

    def set_stage1(self):
        """Stage 1: freeze backbones, train heads only"""
        self.cnn14.freeze_backbone()
        self.ast.freeze_backbone()
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Stage 1: {trainable/1e6:.1f} M trainable (heads only)")

    def set_stage2(self):
        """Stage 2: unfreeze everything for fine-tuning"""
        self.cnn14.unfreeze_all()
        self.ast.unfreeze_last_n_blocks(n=4)   # unfreeze last 4 AST blocks
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Stage 2: {trainable/1e6:.1f} M trainable (partial unfreeze)")


# ── 8. Label smoothing ────────────────────────────────────────────────────────
def smooth_labels(targets, smoothing=CFG.LABEL_SMOOTH):
    return targets * (1 - smoothing) + 0.5 * smoothing


# ── 9. Train / eval loops ─────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, scheduler, scaler, criterion, training=True):
    model.train() if training else model.eval()
    losses, preds, targets = [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch_idx, (mel, label, _) in enumerate(loader):
            mel   = mel.to(CFG.DEVICE)
            label = label.float().to(CFG.DEVICE)

            with torch.cuda.amp.autocast(enabled=(CFG.DEVICE == "cuda")):
                logit = model(mel).squeeze(1)
                lbl   = smooth_labels(label) if training else label
                loss  = criterion(logit, lbl)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()

            losses.append(loss.item())
            preds.extend(torch.sigmoid(logit).detach().cpu().numpy())
            targets.extend(label.cpu().numpy())

            if training and batch_idx % 30 == 0:
                print(f"    batch {batch_idx}/{len(loader)} | loss={loss.item():.4f}")

    preds    = np.array(preds)
    targets  = np.array(targets)
    pred_bin = (preds >= 0.5).astype(int)
    return (np.mean(losses),
            f1_score(targets, pred_bin),
            roc_auc_score(targets, preds),
            accuracy_score(targets, pred_bin),
            preds, targets)


# ── 10. Staged training ───────────────────────────────────────────────────────
def train_staged(model, train_loader, val_loader):
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([1.2]).to(CFG.DEVICE)
    )
    scaler  = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE == "cuda"))
    best_f1 = 0.0
    patience_c = 0
    history = []
    global_epoch = 0

    # ── Stage 1: heads only ──────────────────────────────────────────────────
    print("\n" + "="*50)
    print("STAGE 1 — Training classifier heads only")
    print("="*50)
    model.set_stage1()

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=CFG.STAGE1_LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = OneCycleLR(optimizer, max_lr=CFG.STAGE1_LR,
                           steps_per_epoch=len(train_loader),
                           epochs=CFG.STAGE1_EPOCHS, pct_start=0.2)

    for epoch in range(1, CFG.STAGE1_EPOCHS + 1):
        global_epoch += 1
        t0 = time.time()
        print(f"\n── Epoch {global_epoch} [Stage 1] ──────────────────")
        tr_loss, tr_f1, tr_auc, _, _, _ = run_epoch(model, train_loader, optimizer, scheduler, scaler, criterion, training=True)
        va_loss, va_f1, va_auc, va_acc, _, _ = run_epoch(model, val_loader, None, None, scaler, criterion, training=False)
        print(f"  TRAIN -> loss={tr_loss:.4f} | F1={tr_f1:.4f} | AUC={tr_auc:.4f}")
        print(f"  VAL   -> loss={va_loss:.4f} | F1={va_f1:.4f} | AUC={va_auc:.4f} | Acc={va_acc:.4f} | {time.time()-t0:.0f}s")
        history.append({"epoch": global_epoch, "stage": 1, "tr_f1": tr_f1, "va_f1": va_f1})
        if va_f1 > best_f1:
            best_f1 = va_f1
            torch.save({"epoch": global_epoch, "model_state": model.state_dict(),
                        "val_f1": va_f1, "val_auc": va_auc, "val_acc": va_acc},
                       f"{CFG.OUTPUT_DIR}/best_ensemble.pt")
            print(f"  * New best F1={best_f1:.4f} saved.")

    # ── Stage 2: partial unfreeze ────────────────────────────────────────────
    print("\n" + "="*50)
    print("STAGE 2 — Fine-tuning with partial unfreeze")
    print("="*50)
    model.set_stage2()
    patience_c = 0

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=CFG.STAGE2_LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG.STAGE2_EPOCHS, eta_min=1e-6)

    for epoch in range(1, CFG.STAGE2_EPOCHS + 1):
        global_epoch += 1
        t0 = time.time()
        print(f"\n── Epoch {global_epoch} [Stage 2] ──────────────────")
        tr_loss, tr_f1, tr_auc, _, _, _ = run_epoch(model, train_loader, optimizer, scheduler, scaler, criterion, training=True)
        va_loss, va_f1, va_auc, va_acc, _, _ = run_epoch(model, val_loader, None, None, scaler, criterion, training=False)
        scheduler.step()
        print(f"  TRAIN -> loss={tr_loss:.4f} | F1={tr_f1:.4f} | AUC={tr_auc:.4f}")
        print(f"  VAL   -> loss={va_loss:.4f} | F1={va_f1:.4f} | AUC={va_auc:.4f} | Acc={va_acc:.4f} | {time.time()-t0:.0f}s")
        history.append({"epoch": global_epoch, "stage": 2, "tr_f1": tr_f1, "va_f1": va_f1})
        if va_f1 > best_f1:
            best_f1, patience_c = va_f1, 0
            torch.save({"epoch": global_epoch, "model_state": model.state_dict(),
                        "val_f1": va_f1, "val_auc": va_auc, "val_acc": va_acc},
                       f"{CFG.OUTPUT_DIR}/best_ensemble.pt")
            print(f"  * New best F1={best_f1:.4f} saved.")
        else:
            patience_c += 1
            print(f"  No improvement ({patience_c}/{CFG.PATIENCE})")
            if patience_c >= CFG.PATIENCE:
                print("  Early stopping.")
                break

    pd.DataFrame(history).to_csv(f"{CFG.OUTPUT_DIR}/training_history.csv", index=False)
    print(f"\nTraining complete. Best Val F1 = {best_f1:.4f}")
    return best_f1


# ── 11. Test Evaluation ───────────────────────────────────────────────────────
def evaluate_test(model, test_loader):
    ckpt = torch.load(f"{CFG.OUTPUT_DIR}/best_ensemble.pt",
                      map_location=CFG.DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    print(f"\n-- Test Evaluation (best epoch={ckpt['epoch']}, val F1={ckpt['val_f1']:.4f}) --")

    criterion = nn.BCEWithLogitsLoss()
    _, _, auc, _, preds, targets = run_epoch(model, test_loader, None, None,
                                              torch.cuda.amp.GradScaler(), criterion, training=False)
    # Threshold search
    best_thresh, best_f1_t = 0.5, 0.0
    print("\nThreshold search:")
    for thresh in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
        f1_t = f1_score(targets, (preds >= thresh).astype(int))
        marker = " <-- best" if f1_t > best_f1_t else ""
        print(f"  thresh={thresh} -> F1={f1_t:.4f}{marker}")
        if f1_t > best_f1_t:
            best_f1_t, best_thresh = f1_t, thresh

    pred_bin = (preds >= best_thresh).astype(int)
    acc = accuracy_score(targets, pred_bin)
    print(f"\nFinal at threshold={best_thresh}:")
    print(f"  F1={best_f1_t:.4f} | AUC={auc:.4f} | Acc={acc:.4f}")
    print(classification_report(targets, pred_bin, target_names=["noncooking", "cooking"]))

    pd.DataFrame({"prob_cooking": preds, "pred_label": pred_bin,
                  "true_label": targets.astype(int)
                  }).to_csv(f"{CFG.OUTPUT_DIR}/test_predictions.csv", index=False)
    return {"f1": best_f1_t, "auc": auc, "accuracy": acc, "threshold": best_thresh}


# ── 12. Run ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("Phase 7 v4 -- CNN14 + AST (Staged Unfreezing)")
print("Samsung PRISM | kitpri_v2")
print("=" * 60)

print("Loading datasets...")
train_loader, val_loader, test_loader = get_loaders()

print("\nBuilding model...")
model = CNN14_AST_Ensemble().to(CFG.DEVICE)

best_f1      = train_staged(model, train_loader, val_loader)
test_results = evaluate_test(model, test_loader)

print("\n" + "=" * 60)
print("Phase 7 v4 COMPLETE")
print(f"  Best Val F1  : {best_f1:.4f}")
print(f"  Test F1      : {test_results['f1']:.4f}")
print(f"  Test AUC     : {test_results['auc']:.4f}")
print(f"  Test Accuracy: {test_results['accuracy']:.4f}")
print(f"  Threshold    : {test_results['threshold']}")
print(f"  F1 > 0.95?   : {'TARGET MET' if test_results['f1'] >= 0.95 else 'Below target'}")
print("=" * 60)

# the session kept expiring and i have to re start the whole run, 
# so now Phase 7 V5 with added checkpoints so that the re run resumes fron where is stopped 


In [ ]:
!pip install -q timm panns-inference

In [ ]:
# =============================================================================
# PHASE 7 v5 — CNN14 + AST Ensemble (Staged + RESUMABLE)
# Samsung PRISM — Kitchen Audio Classification (kitpri_v2)
# 
# RESUME LOGIC: If Kaggle session times out, just re-run the cell.
# The script auto-detects the latest checkpoint and resumes from there.
# =============================================================================
# CELL 1 (separate): !pip install -q timm panns-inference
# CELL 2: Paste everything below into ONE cell and run
# =============================================================================

import os, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR

import librosa
import timm
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, classification_report

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True


# ── 1. Config ─────────────────────────────────────────────────────────────────
class CFG:
    DATA_ROOT    = "/kaggle/input/datasets/ayushalia/kitpri-v2/kitpri_v2"
    AUDIO_DIR    = f"{DATA_ROOT}/audio_32k"
    TRAIN_CSV    = f"{DATA_ROOT}/metadata/train.csv"
    VAL_CSV      = f"{DATA_ROOT}/metadata/val.csv"
    TEST_CSV     = f"{DATA_ROOT}/metadata/test.csv"
    OUTPUT_DIR   = "/kaggle/working/phase7_outputs"

    SR           = 32000
    DURATION     = 10
    N_SAMPLES    = SR * DURATION
    N_FFT        = 1024
    HOP_LENGTH   = 320
    N_MELS       = 64
    F_MIN        = 50
    F_MAX        = 14000

    STAGE1_EPOCHS = 8
    STAGE1_LR     = 3e-4
    STAGE2_EPOCHS = 40
    STAGE2_LR     = 5e-5

    BATCH_SIZE   = 24
    WEIGHT_DECAY = 5e-3
    NUM_WORKERS  = 2
    SEED         = 42

    CNN14_WEIGHT = 0.45
    AST_WEIGHT   = 0.55
    PATIENCE     = 12
    LABEL_SMOOTH = 0.05

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Checkpoint files
    LATEST_CKPT = f"{OUTPUT_DIR}/latest_ckpt.pt"   # for resume
    BEST_CKPT   = f"{OUTPUT_DIR}/best_ensemble.pt" # best F1
    HISTORY_CSV = f"{OUTPUT_DIR}/training_history.csv"

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print(f"Device : {CFG.DEVICE}")
print(f"Root   : {CFG.DATA_ROOT}")


# ── 2. Reproducibility ────────────────────────────────────────────────────────
def seed_everything(seed=CFG.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()


# ── 3. Filter CSVs ────────────────────────────────────────────────────────────
print("\nFiltering CSVs to existing files only...")
for attr, csv_path in [("TRAIN_CSV", CFG.TRAIN_CSV),
                        ("VAL_CSV",   CFG.VAL_CSV),
                        ("TEST_CSV",  CFG.TEST_CSV)]:
    df     = pd.read_csv(csv_path)
    before = len(df)
    df     = df[df["file_path"].apply(
                    lambda p: os.path.exists(f"{CFG.DATA_ROOT}/{p}")
                )].reset_index(drop=True)
    after  = len(df)
    out    = f"/kaggle/working/{os.path.basename(csv_path)}"
    df.to_csv(out, index=False)
    setattr(CFG, attr, out)
    print(f"  {os.path.basename(csv_path)}: {before} -> {after} rows")
print("Done.\n")


# ── 4. Dataset ────────────────────────────────────────────────────────────────
class KitpriDataset(Dataset):
    def __init__(self, csv_path, augment=False):
        self.df        = pd.read_csv(csv_path)
        self.root      = Path(CFG.DATA_ROOT)
        self.augment   = augment
        self.n_samples = CFG.N_SAMPLES
        print(f"  {Path(csv_path).name}: {len(self.df)} clips | "
              f"cooking={(self.df['label']==1).sum()} | "
              f"noncooking={(self.df['label']==0).sum()}")

    def __len__(self): return len(self.df)

    def _load_audio(self, path):
        wav, _ = librosa.load(str(path), sr=CFG.SR, mono=True, duration=CFG.DURATION)
        if len(wav) < self.n_samples:
            wav = np.pad(wav, (0, self.n_samples - len(wav)))
        else:
            wav = wav[:self.n_samples]
        return wav.astype(np.float32)

    def _to_melspec(self, wav):
        mel = librosa.feature.melspectrogram(
            y=wav, sr=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, fmin=CFG.F_MIN, fmax=CFG.F_MAX)
        mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        return (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    def _augment(self, wav):
        wav = wav * np.random.uniform(0.7, 1.3)
        wav = np.roll(wav, np.random.randint(-CFG.SR // 2, CFG.SR // 2))
        wav = wav + np.random.randn(len(wav)).astype(np.float32) * 0.005
        return np.clip(wav, -1.0, 1.0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = self._load_audio(self.root / row["file_path"])
        if self.augment: wav = self._augment(wav)
        return (torch.tensor(self._to_melspec(wav)).unsqueeze(0),
                int(row["label"]),
                str(self.root / row["file_path"]))


def get_loaders():
    train_ds = KitpriDataset(CFG.TRAIN_CSV, augment=True)
    val_ds   = KitpriDataset(CFG.VAL_CSV,   augment=False)
    test_ds  = KitpriDataset(CFG.TEST_CSV,  augment=False)
    return (
        DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                   num_workers=CFG.NUM_WORKERS, pin_memory=True),
        DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                   num_workers=CFG.NUM_WORKERS, pin_memory=True),
        DataLoader(test_ds,  batch_size=CFG.BATCH_SIZE, shuffle=False,
                   num_workers=CFG.NUM_WORKERS, pin_memory=True),
    )


# ── 5. CNN14 ──────────────────────────────────────────────────────────────────
class CNN14(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        def conv_block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
        self.bn0    = nn.BatchNorm2d(1)
        self.block1 = conv_block(1,    64)
        self.block2 = conv_block(64,   128)
        self.block3 = conv_block(128,  256)
        self.block4 = conv_block(256,  512)
        self.block5 = conv_block(512,  1024)
        self.block6 = conv_block(1024, 2048)
        self.pool   = nn.AvgPool2d(2, 2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.fc1    = nn.Linear(2048, 512)
        self.drop   = nn.Dropout(0.5)
        self.fc_out = nn.Linear(512, 1)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):     nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):    nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
        if pretrained: self._load_panns_weights()

    def _load_panns_weights(self):
        try:
            from panns_inference import AudioTagging
            at = AudioTagging(checkpoint_path=None, device="cpu")
            our_sd = self.state_dict()
            matched = 0
            for k, v in at.model.state_dict().items():
                if k in our_sd and our_sd[k].shape == v.shape:
                    our_sd[k] = v; matched += 1
            self.load_state_dict(our_sd, strict=False)
            print(f"  [CNN14] PANNs weights matched: {matched}")
        except Exception:
            print(f"  [CNN14] PANNs unavailable, training from scratch")

    def freeze_backbone(self):
        for n, p in self.named_parameters():
            if "fc1" not in n and "fc_out" not in n: p.requires_grad = False

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True

    def forward(self, x):
        x = self.bn0(x)
        for b in [self.block1, self.block2, self.block3, self.block4, self.block5]:
            x = self.pool(b(x))
        x = self.block6(x)
        x = self.gap(x).flatten(1)
        return self.fc_out(self.drop(F.relu(self.fc1(x))))


# ── 6. AST ────────────────────────────────────────────────────────────────────
class ASTModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "deit_small_patch16_224",
            pretrained=pretrained, num_classes=0, global_pool="token")
        ed = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(ed), nn.Linear(ed, 256), nn.GELU(),
            nn.Dropout(0.4), nn.Linear(256, 1))
        print(f"  [AST] DeiT-Small | embed_dim={ed}")

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_last_n_blocks(self, n=4):
        for p in self.backbone.parameters(): p.requires_grad = False
        for b in self.backbone.blocks[-n:]:
            for p in b.parameters(): p.requires_grad = True
        for p in self.backbone.norm.parameters(): p.requires_grad = True

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        return self.head(self.backbone(x))


# ── 7. Ensemble ───────────────────────────────────────────────────────────────
class CNN14_AST_Ensemble(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn14 = CNN14(pretrained=True)
        self.ast   = ASTModel(pretrained=True)

    def forward(self, x):
        return CFG.CNN14_WEIGHT * self.cnn14(x) + CFG.AST_WEIGHT * self.ast(x)

    def set_stage1(self):
        self.cnn14.freeze_backbone(); self.ast.freeze_backbone()
        t = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Stage 1: {t/1e6:.1f} M trainable (heads only)")

    def set_stage2(self):
        self.cnn14.unfreeze_all()
        self.ast.unfreeze_last_n_blocks(n=4)
        t = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Stage 2: {t/1e6:.1f} M trainable (partial unfreeze)")


# ── 8. Label smoothing ────────────────────────────────────────────────────────
def smooth_labels(t, s=CFG.LABEL_SMOOTH):
    return t * (1 - s) + 0.5 * s


# ── 9. Train / eval ───────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, scheduler, scaler, criterion, training=True):
    model.train() if training else model.eval()
    losses, preds, targets = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for bi, (mel, label, _) in enumerate(loader):
            mel, label = mel.to(CFG.DEVICE), label.float().to(CFG.DEVICE)
            with torch.cuda.amp.autocast(enabled=(CFG.DEVICE=="cuda")):
                logit = model(mel).squeeze(1)
                loss  = criterion(logit, smooth_labels(label) if training else label)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None: scheduler.step()
            losses.append(loss.item())
            preds.extend(torch.sigmoid(logit).detach().cpu().numpy())
            targets.extend(label.cpu().numpy())
            if training and bi % 30 == 0:
                print(f"    batch {bi}/{len(loader)} | loss={loss.item():.4f}")
    preds, targets = np.array(preds), np.array(targets)
    pred_bin = (preds >= 0.5).astype(int)
    return (np.mean(losses), f1_score(targets, pred_bin),
            roc_auc_score(targets, preds), accuracy_score(targets, pred_bin),
            preds, targets)


# ── 10. Checkpoint helpers ────────────────────────────────────────────────────
def save_checkpoint(model, optimizer, scheduler, scaler, epoch, stage,
                    best_f1, patience_c, history, is_best=False):
    """Save full state every epoch for resume."""
    state = {
        "epoch":       epoch,
        "stage":       stage,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "sched_state": scheduler.state_dict() if scheduler else None,
        "scaler_state": scaler.state_dict(),
        "best_f1":     best_f1,
        "patience_c":  patience_c,
        "history":     history,
    }
    torch.save(state, CFG.LATEST_CKPT)
    if is_best:
        torch.save(state, CFG.BEST_CKPT)


def load_checkpoint():
    """Returns state dict if checkpoint exists, else None."""
    if os.path.exists(CFG.LATEST_CKPT):
        print(f"\n✓ Resuming from existing checkpoint: {CFG.LATEST_CKPT}")
        ckpt = torch.load(CFG.LATEST_CKPT, map_location=CFG.DEVICE, weights_only=False)
        print(f"  Last completed: epoch {ckpt['epoch']} (stage {ckpt['stage']})")
        print(f"  Best F1 so far: {ckpt['best_f1']:.4f}")
        return ckpt
    return None


# ── 11. Staged training with resume ───────────────────────────────────────────
def train_staged(model, train_loader, val_loader):
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.2]).to(CFG.DEVICE))
    scaler    = torch.cuda.amp.GradScaler(enabled=(CFG.DEVICE=="cuda"))

    # Try to resume
    ckpt = load_checkpoint()
    start_epoch = ckpt["epoch"] + 1 if ckpt else 1
    start_stage = ckpt["stage"] if ckpt else 1
    best_f1     = ckpt["best_f1"] if ckpt else 0.0
    patience_c  = ckpt["patience_c"] if ckpt else 0
    history     = ckpt["history"] if ckpt else []

    if ckpt:
        model.load_state_dict(ckpt["model_state"])

    total_epochs = CFG.STAGE1_EPOCHS + CFG.STAGE2_EPOCHS

    # ── STAGE 1 ──────────────────────────────────────────────────────────────
    if start_stage == 1 and start_epoch <= CFG.STAGE1_EPOCHS:
        print("\n" + "="*50)
        print("STAGE 1 — Training classifier heads only")
        print("="*50)
        model.set_stage1()
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(trainable, lr=CFG.STAGE1_LR, weight_decay=CFG.WEIGHT_DECAY)
        scheduler = OneCycleLR(optimizer, max_lr=CFG.STAGE1_LR,
                               steps_per_epoch=len(train_loader),
                               epochs=CFG.STAGE1_EPOCHS, pct_start=0.2)
        if ckpt and ckpt["stage"] == 1:
            optimizer.load_state_dict(ckpt["optim_state"])
            if ckpt["sched_state"]: scheduler.load_state_dict(ckpt["sched_state"])
            scaler.load_state_dict(ckpt["scaler_state"])

        for epoch in range(start_epoch, CFG.STAGE1_EPOCHS + 1):
            t0 = time.time()
            print(f"\n── Epoch {epoch}/{total_epochs} [Stage 1] ─────────")
            tr_loss, tr_f1, tr_auc, _, _, _ = run_epoch(model, train_loader, optimizer, scheduler, scaler, criterion, True)
            va_loss, va_f1, va_auc, va_acc, _, _ = run_epoch(model, val_loader, None, None, scaler, criterion, False)
            print(f"  TRAIN -> loss={tr_loss:.4f} | F1={tr_f1:.4f} | AUC={tr_auc:.4f}")
            print(f"  VAL   -> loss={va_loss:.4f} | F1={va_f1:.4f} | AUC={va_auc:.4f} | Acc={va_acc:.4f} | {time.time()-t0:.0f}s")
            history.append({"epoch": epoch, "stage": 1, "tr_f1": tr_f1, "va_f1": va_f1, "va_auc": va_auc})
            is_best = va_f1 > best_f1
            if is_best:
                best_f1 = va_f1
                print(f"  * New best F1={best_f1:.4f}")
            save_checkpoint(model, optimizer, scheduler, scaler, epoch, 1,
                            best_f1, patience_c, history, is_best=is_best)
            pd.DataFrame(history).to_csv(CFG.HISTORY_CSV, index=False)

        start_epoch = CFG.STAGE1_EPOCHS + 1
        start_stage = 2

    # ── STAGE 2 ──────────────────────────────────────────────────────────────
    if start_stage == 2:
        print("\n" + "="*50)
        print("STAGE 2 — Fine-tuning with partial unfreeze")
        print("="*50)
        model.set_stage2()
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(trainable, lr=CFG.STAGE2_LR, weight_decay=CFG.WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=CFG.STAGE2_EPOCHS, eta_min=1e-6)
        if ckpt and ckpt["stage"] == 2:
            optimizer.load_state_dict(ckpt["optim_state"])
            if ckpt["sched_state"]: scheduler.load_state_dict(ckpt["sched_state"])
            scaler.load_state_dict(ckpt["scaler_state"])

        stage2_start = max(start_epoch, CFG.STAGE1_EPOCHS + 1)
        for epoch in range(stage2_start, total_epochs + 1):
            t0 = time.time()
            print(f"\n── Epoch {epoch}/{total_epochs} [Stage 2] ─────────")
            tr_loss, tr_f1, tr_auc, _, _, _ = run_epoch(model, train_loader, optimizer, scheduler, scaler, criterion, True)
            va_loss, va_f1, va_auc, va_acc, _, _ = run_epoch(model, val_loader, None, None, scaler, criterion, False)
            print(f"  TRAIN -> loss={tr_loss:.4f} | F1={tr_f1:.4f} | AUC={tr_auc:.4f}")
            print(f"  VAL   -> loss={va_loss:.4f} | F1={va_f1:.4f} | AUC={va_auc:.4f} | Acc={va_acc:.4f} | {time.time()-t0:.0f}s")
            history.append({"epoch": epoch, "stage": 2, "tr_f1": tr_f1, "va_f1": va_f1, "va_auc": va_auc})
            is_best = va_f1 > best_f1
            if is_best:
                best_f1, patience_c = va_f1, 0
                print(f"  * New best F1={best_f1:.4f}")
            else:
                patience_c += 1
                print(f"  No improvement ({patience_c}/{CFG.PATIENCE})")
            save_checkpoint(model, optimizer, scheduler, scaler, epoch, 2,
                            best_f1, patience_c, history, is_best=is_best)
            pd.DataFrame(history).to_csv(CFG.HISTORY_CSV, index=False)
            if patience_c >= CFG.PATIENCE:
                print("  Early stopping triggered.")
                break

    print(f"\nTraining complete. Best Val F1 = {best_f1:.4f}")
    return best_f1


# ── 12. Test ──────────────────────────────────────────────────────────────────
def evaluate_test(model, test_loader):
    ckpt = torch.load(CFG.BEST_CKPT, map_location=CFG.DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    print(f"\n-- Test Evaluation (epoch={ckpt['epoch']}, val F1={ckpt['best_f1']:.4f}) --")
    criterion = nn.BCEWithLogitsLoss()
    _, _, auc, _, preds, targets = run_epoch(model, test_loader, None, None,
                                              torch.cuda.amp.GradScaler(), criterion, False)
    best_thresh, best_f1_t = 0.5, 0.0
    print("\nThreshold search:")
    for t in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
        f = f1_score(targets, (preds >= t).astype(int))
        m = " <-- best" if f > best_f1_t else ""
        print(f"  thresh={t} -> F1={f:.4f}{m}")
        if f > best_f1_t: best_f1_t, best_thresh = f, t
    pred_bin = (preds >= best_thresh).astype(int)
    acc = accuracy_score(targets, pred_bin)
    print(f"\nFinal at threshold={best_thresh}: F1={best_f1_t:.4f} | AUC={auc:.4f} | Acc={acc:.4f}")
    print(classification_report(targets, pred_bin, target_names=["noncooking", "cooking"]))
    pd.DataFrame({"prob_cooking": preds, "pred_label": pred_bin,
                  "true_label": targets.astype(int)}
                  ).to_csv(f"{CFG.OUTPUT_DIR}/test_predictions.csv", index=False)
    return {"f1": best_f1_t, "auc": auc, "accuracy": acc, "threshold": best_thresh}


# ── 13. Run ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("Phase 7 v5 -- CNN14 + AST (Staged + Resumable)")
print("Samsung PRISM | kitpri_v2")
print("=" * 60)

print("Loading datasets...")
train_loader, val_loader, test_loader = get_loaders()

print("\nBuilding model...")
model = CNN14_AST_Ensemble().to(CFG.DEVICE)

best_f1      = train_staged(model, train_loader, val_loader)
test_results = evaluate_test(model, test_loader)

print("\n" + "=" * 60)
print("Phase 7 v5 COMPLETE")
print(f"  Best Val F1  : {best_f1:.4f}")
print(f"  Test F1      : {test_results['f1']:.4f}")
print(f"  Test AUC     : {test_results['auc']:.4f}")
print(f"  Test Accuracy: {test_results['accuracy']:.4f}")
print(f"  F1 > 0.95?   : {'TARGET MET' if test_results['f1'] >= 0.95 else 'Below target'}")
print("=" * 60)



---

## Why we're dropping CNN14 from the ensemble

### What CNN14 is
CNN14 is a 14-layer convolutional neural network from the PANNs family (Pretrained Audio Neural Networks, Kong et al. 2020). It was designed to process log-mel spectrograms using 2D convolutions, similar to how image CNNs work. It was originally trained on AudioSet (527 classes, ~2 million clips).

### Why CNN14 is failing on our dataset

**Problem 1 — No usable pretrained weights**
PANNs CNN14 requires the `panns-inference` package to load its pretrained weights. On Kaggle, this either fails to install or loads incorrectly. In every run we saw:
```
[CNN14] PANNs weights: 3 matched, 81 skipped
```
Only 3 out of 84 weight layers transferred. The remaining 81 layers trained from **random initialization**. This means CNN14 was essentially training from scratch with 2428 samples — far too little data for a model with 97M parameters.

**Problem 2 — Wrong parameter count for dataset size**
CNN14 alone has ~97M parameters. Our training set has 2428 samples. That's roughly **40,000 parameters per sample** — a ratio that guarantees memorization rather than generalization. This is exactly what we saw: Train F1 → 1.0, Val F1 stuck at 0.85.

**Problem 3 — It actively hurts the ensemble**
In the ensemble, CNN14's weak predictions (random-init model) are averaged with AST's strong predictions. This drags the ensemble score down. Removing CNN14 and using AST alone gives the stronger model full control.

---

## What AST is and why it works better

AST (Audio Spectrogram Transformer, Gong et al. 2021) treats a mel spectrogram as an image and passes it through a Vision Transformer (ViT) backbone. We use DeiT-Small pretrained on ImageNet as the backbone.

**Key advantage — real pretrained weights**
DeiT-Small downloads 88MB of genuine ImageNet-pretrained weights via `timm`. Every single layer starts from a meaningful initialization. Even though ImageNet contains photos (not audio), the low-level feature detectors (edges, textures, patterns) transfer well to spectrograms.

**Smaller and better suited**
DeiT-Small has ~22M parameters. With 2428 training samples that's ~9,000 parameters per sample — a much healthier ratio that allows generalization without memorization.

---

## Side-by-side comparison for your documentation

| Property | CNN14 | AST (DeiT-Small) |
|---|---|---|
| Architecture | 14-layer 2D CNN | Vision Transformer (ViT) |
| Input | Log-mel spectrogram | Log-mel spectrogram (resized to 224×224) |
| Pretrained on | AudioSet (527 classes) | ImageNet (1000 classes) |
| Pretrained weights available on Kaggle | ✗ No (3/84 layers only) | ✓ Yes (full 88MB download) |
| Parameters | ~97M | ~22M |
| Parameters per sample (2428 clips) | ~40,000 | ~9,000 |
| Overfitting risk | Very high | Moderate |
| Individual F1 on kitpri_v2 | ~0.80 | ~0.93 |
| Contribution to ensemble | Negative (drags down AST) | Positive |
| Suitable for dataset size | No | Yes |

---

## Decision

CNN14 will be **re-evaluated after data expansion** (if the dataset grows to 10,000+ clips). At that scale, the pretrained weights issue can be solved properly and the parameter ratio becomes acceptable. For now, **AST alone is the correct choice** for kitpri_v2 at 2428 training samples.


In [ ]:
#################################################################################################


# AST (Audio Spectrogram Transformer) — Why It Works Better

## What is AST?

**AST (Audio Spectrogram Transformer)** is a deep learning model specifically designed for **audio classification tasks**. It was introduced in the paper **"AST: Audio Spectrogram Transformer" (Gong et al., 2021)**.

Unlike traditional audio models that rely on Convolutional Neural Networks (CNNs), AST uses a **Transformer architecture**, similar to the Vision Transformer (ViT) used in computer vision.

The core idea is:

```text
Audio
  ↓
Mel Spectrogram
  ↓
Transformer
  ↓
Prediction
````

Instead of processing raw audio directly, AST first converts audio into a visual representation called a **Mel Spectrogram** and then treats it as an image.

---

## Why Convert Audio into a Spectrogram?

Raw audio is simply a sequence of amplitude values over time:

```text
[0.12, 0.15, -0.03, 0.21, ...]
```

This format is difficult for image-based deep learning models to interpret effectively.

Therefore, the audio is transformed into a **Mel Spectrogram**, where:

* X-axis = Time
* Y-axis = Frequency
* Color Intensity = Energy of the sound

Different sounds create different visual patterns:

| Sound        | Spectrogram Pattern          |
| ------------ | ---------------------------- |
| Bird Chirp   | Thin rising lines            |
| Engine Noise | Horizontal bands             |
| Rain         | Random textures              |
| Human Speech | Complex frequency structures |

This effectively converts:

```text
Audio → Image
```

allowing powerful image-based architectures to be applied to audio problems.

---

## How AST Processes Audio

The complete AST pipeline is:

```text
Audio File
      ↓
Mel Spectrogram
      ↓
Split into Patches
      ↓
Transformer Encoder
      ↓
Classification Head
      ↓
Predicted Class
```

Example:

```text
dog_bark.wav
      ↓
Mel Spectrogram
      ↓
Transformer
      ↓
Dog Bark
```

---

## What is a Transformer?

Transformers were originally developed for Natural Language Processing (NLP) and later adapted for computer vision.

Popular transformer models include:

* BERT
* GPT
* T5
* Vision Transformer (ViT)

Transformers use a mechanism called **Self-Attention**, which allows the model to determine which parts of the input are most important and how different regions relate to each other.

Unlike CNNs, transformers can capture long-range relationships across the entire input.

---

## CNN vs Transformer

### CNN Approach

CNNs process data using convolution filters:

```text
Image
  ↓
Convolution
  ↓
Feature Map
```

CNNs primarily focus on:

* Local patterns
* Edges
* Corners
* Textures

Information is learned gradually through multiple convolution layers.

---

### Transformer Approach

Transformers process inputs using attention:

```text
Patch A ↔ Patch B
Patch A ↔ Patch C
Patch B ↔ Patch D
```

Benefits:

* Captures global context
* Learns relationships between distant regions
* Understands complex patterns more effectively

For audio classification, this is useful because important sound events may occur at different times in the recording, and transformers can connect those events directly.

---

## How AST Uses Vision Transformers

AST converts the spectrogram into small image patches.

For example:

```text
Spectrogram
      ↓
16×16 Patches
      ↓
Patch Tokens
      ↓
Transformer
```

Each patch becomes a token, similar to how words become tokens in NLP.

The transformer then learns relationships between all patches simultaneously.

This allows AST to understand:

* Frequency patterns
* Temporal patterns
* Global audio structure

all at the same time.

---

## Backbone Used in Our Project

Our implementation uses:

**DeiT-Small (Data-efficient Image Transformer Small)**

The DeiT-Small model acts as the backbone responsible for feature extraction.

AST is built on top of this pretrained Vision Transformer.

---

## What is Pretraining?

Pretraining means training a model on a very large dataset before fine-tuning it on a smaller target dataset.

Instead of starting from random weights, the model starts with useful learned knowledge.

```text
Large Dataset
      ↓
Pretraining
      ↓
Learn General Features
      ↓
Fine-Tune on Target Dataset
```

---

## What Dataset Was DeiT-Small Pretrained On?

DeiT-Small was pretrained on **ImageNet**, which contains approximately:

* 1.2 million images
* 1000 object categories

Examples include:

* Dogs
* Cars
* Birds
* Buildings
* Trees
* People

Through this training, the model learns general visual representations.

---

## What Does the Model Learn During Pretraining?

### Early Layers Learn

* Edges
* Lines
* Corners
* Curves
* Textures

### Middle Layers Learn

* Shapes
* Object Parts
* Complex Patterns

### Later Layers Learn

* High-level object concepts

The most transferable knowledge comes from the early and middle layers.

---

## Why Do ImageNet Features Transfer to Audio?

This is one of the most important concepts behind AST.

Although ImageNet contains photographs and not audio, spectrograms are still images.

Spectrograms contain:

* Edges
* Textures
* Shapes
* Frequency bands
* Repeating visual patterns

Examples:

### Bird Chirp

```text
////
```

### Engine Noise

```text
========
```

### Rain

```text
:::::::::
```

These visual structures are similar to the patterns learned during ImageNet pretraining.

Therefore:

```text
Image Features
       ↓
Transfer Learning
       ↓
Spectrogram Features
```

This allows AST to benefit from ImageNet knowledge even though the target task is audio classification.

---

# Why AST Works Better Than CNN14

## 1. Real Pretrained Weights Loaded Successfully

### CNN14

Observed during training:

```text
[CNN14] PANNs weights: 3 matched, 81 skipped
```

Out of 84 layers:

* Only 3 layers received pretrained weights
* 81 layers remained randomly initialized

As a result, CNN14 was effectively training from scratch.

---

### AST

For AST:

* All DeiT-Small pretrained weights load correctly
* Every layer starts with meaningful learned representations

This provides a significant advantage when working with limited training data.

---

## 2. Smaller and More Suitable Model Size

### CNN14

```text
~97 Million Parameters
```

### AST (DeiT-Small)

```text
~22 Million Parameters
```

AST is more than four times smaller.

Benefits:

* Easier to train
* Requires less data
* Lower risk of overfitting
* Better generalization

---

## 3. Better Dataset-to-Model Capacity Ratio

Dataset size:

```text
2428 audio samples
```

### CNN14

```text
97,000,000 / 2428
≈ 40,000 parameters per sample
```

### AST

```text
22,000,000 / 2428
≈ 9,000 parameters per sample
```

This ratio is not a strict rule, but it provides intuition about model capacity relative to available data.

A lower ratio generally indicates:

* Lower memorization risk
* Better generalization potential

---

## 4. Lower Overfitting Risk

CNN14 produced:

```text
Train F1 = 1.00
Validation F1 ≈ 0.85
```

This indicates:

* Perfect performance on training data
* Significantly lower performance on unseen data

A classic sign of overfitting.

AST generalizes better because it combines:

* Effective pretraining
* Smaller model size
* Better architecture

---

## 5. Better Feature Learning Through Attention

CNNs primarily learn:

```text
Local Features
```

Transformers learn:

```text
Local Features
+
Global Relationships
```

This is particularly important for audio because meaningful sound events can occur far apart in time.

Self-attention allows AST to connect these distant regions directly.

---

# Final Conclusion

AST (Audio Spectrogram Transformer) converts audio into Mel Spectrograms and processes them using a Vision Transformer backbone. In our project, AST uses a DeiT-Small model pretrained on ImageNet, allowing every layer to start with meaningful learned features rather than random initialization. Although ImageNet contains natural images rather than audio, the learned visual representations transfer effectively because spectrograms are image-like representations of sound. AST contains approximately 22 million parameters, significantly fewer than CNN14's 97 million parameters, making it much better suited to our dataset of 2428 audio samples. Combined with successful pretraining, lower overfitting risk, and stronger feature extraction through self-attention, AST consistently provides better generalization and overall performance than CNN14 on our audio classification task.

```
```


# PHASE 7 v6 (AST ONLY)

In [ ]:
!pip install -q timm==1.0.11 torchaudio

In [ ]:
import os, json, time, random, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import timm
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

CFG = dict(
    data_root   = "/kaggle/input/datasets/ayushalia/kitpri-v2/kitpri_v2",
    work_dir    = "/kaggle/working",
    sr          = 32000,
    clip_sec    = 5.0,
    n_mels      = 128,
    n_fft       = 1024,
    hop         = 320,        # 100 fps at 32k
    fmin        = 50,
    fmax        = 14000,
    img_size    = 224,        # DeiT-Small input
    batch_size  = 32,
    num_workers = 4,
    stage1_epochs = 6,        # head only
    stage2_epochs = 30,       # last 6 blocks unfrozen
    lr_head     = 3e-4,
    lr_backbone = 3e-5,
    weight_decay= 5e-3,
    label_smooth= 0.05,
    pos_weight  = 1.2,        # boost noncooking recall
    mixup_alpha = 0.2,
    spec_time_mask = 40,
    spec_freq_mask = 20,
    grad_clip   = 1.0,
    amp         = True,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["work_dir"], exist_ok=True)
CKPT_LATEST = os.path.join(CFG["work_dir"], "latest_ckpt.pt")
CKPT_BEST   = os.path.join(CFG["work_dir"], "best_ckpt.pt")
print("device:", device)

In [ ]:
ROOT = Path(CFG["data_root"])
META = ROOT / "metadata"

def load_and_filter(split):
    df = pd.read_csv(META / f"{split}.csv")
    # rel path column — adjust if your CSV uses a different name
    path_col = "filepath" if "filepath" in df.columns else ("path" if "path" in df.columns else df.columns[0])
    df["abs_path"] = df[path_col].apply(lambda p: str(ROOT / p) if not str(p).startswith("/") else str(p))
    before = len(df)
    df = df[df["abs_path"].apply(os.path.exists)].reset_index(drop=True)
    print(f"{split}: kept {len(df)}/{before}")
    # label col
    if "label" not in df.columns:
        # binary inferred from folder
        df["label"] = df["abs_path"].apply(lambda p: 1 if "/cooking/" in p else 0)
    return df

train_df = load_and_filter("train")
val_df   = load_and_filter("val")
test_df  = load_and_filter("test")
print("class balance (train):", train_df.label.value_counts().to_dict())

In [ ]:
CLIP_LEN = int(CFG["sr"] * CFG["clip_sec"])

melspec = T.MelSpectrogram(
    sample_rate=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop"],
    n_mels=CFG["n_mels"], f_min=CFG["fmin"], f_max=CFG["fmax"], power=2.0,
).to(device)
amp_to_db = T.AmplitudeToDB(top_db=80).to(device)

class KitchenDS(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.train = train
    def __len__(self): return len(self.df)
    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        if sr != CFG["sr"]:
            wav = torchaudio.functional.resample(wav, sr, CFG["sr"])
        wav = wav.squeeze(0)
        if wav.numel() < CLIP_LEN:
            wav = F.pad(wav, (0, CLIP_LEN - wav.numel()))
        else:
            if self.train:
                start = random.randint(0, wav.numel() - CLIP_LEN)
            else:
                start = (wav.numel() - CLIP_LEN) // 2
            wav = wav[start:start+CLIP_LEN]
        return wav
    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav = self._load(row["abs_path"])
        return wav, torch.tensor(float(row["label"]))

def collate(batch):
    wavs = torch.stack([b[0] for b in batch])
    ys   = torch.stack([b[1] for b in batch])
    return wavs, ys

def wav_to_logmel(wav_batch):
    # wav_batch: (B, T) on device
    m = melspec(wav_batch)           # (B, n_mels, frames)
    m = amp_to_db(m)
    # normalize per-sample
    m = (m - m.mean(dim=(1,2), keepdim=True)) / (m.std(dim=(1,2), keepdim=True) + 1e-6)
    return m

def spec_augment(mel, F_mask=20, T_mask=40, n_freq=2, n_time=2):
    B, M, Tt = mel.shape
    for _ in range(n_freq):
        f = random.randint(0, F_mask)
        f0 = random.randint(0, max(0, M - f))
        mel[:, f0:f0+f, :] = 0
    for _ in range(n_time):
        t = random.randint(0, T_mask)
        t0 = random.randint(0, max(0, Tt - t))
        mel[:, :, t0:t0+t] = 0
    return mel

def mel_to_img(mel):
    # resize to (img_size, img_size) and repeat to 3 channels for DeiT
    mel = mel.unsqueeze(1)  # (B,1,M,T)
    mel = F.interpolate(mel, size=(CFG["img_size"], CFG["img_size"]), mode="bilinear", align_corners=False)
    mel = mel.repeat(1, 3, 1, 1)
    return mel

In [ ]:
class ASTLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "deit_small_patch16_224", pretrained=True, num_classes=0, global_pool="avg"
        )
        d = self.backbone.num_features  # 384
        self.head = nn.Sequential(
            nn.LayerNorm(d), nn.Dropout(0.2), nn.Linear(d, 1)
        )
    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats).squeeze(-1)

def freeze_all_but_head(model):
    for p in model.backbone.parameters(): p.requires_grad = False
    for p in model.head.parameters():     p.requires_grad = True

def unfreeze_last_n_blocks(model, n=6):
    blocks = model.backbone.blocks
    for blk in blocks[-n:]:
        for p in blk.parameters(): p.requires_grad = True
    # also unfreeze final norm
    for p in model.backbone.norm.parameters(): p.requires_grad = True

def mixup(x, y, alpha=0.2):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam

In [ ]:
def make_loaders():
    tr = DataLoader(KitchenDS(train_df, train=True),  batch_size=CFG["batch_size"],
                    shuffle=True,  num_workers=CFG["num_workers"], collate_fn=collate, pin_memory=True, drop_last=True)
    va = DataLoader(KitchenDS(val_df,   train=False), batch_size=CFG["batch_size"],
                    shuffle=False, num_workers=CFG["num_workers"], collate_fn=collate, pin_memory=True)
    te = DataLoader(KitchenDS(test_df,  train=False), batch_size=CFG["batch_size"],
                    shuffle=False, num_workers=CFG["num_workers"], collate_fn=collate, pin_memory=True)
    return tr, va, te

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for wav, y in loader:
        wav, y = wav.to(device, non_blocking=True), y.to(device)
        mel = wav_to_logmel(wav)
        img = mel_to_img(mel)
        with torch.cuda.amp.autocast(enabled=CFG["amp"]):
            logits = model(img)
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.cpu().numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    preds = (ps >= 0.5).astype(int)
    f1 = f1_score(ys, preds)
    try: auc = roc_auc_score(ys, ps)
    except: auc = float("nan")
    p, r, _, _ = precision_recall_fscore_support(ys, preds, average="binary")
    cm = confusion_matrix(ys, preds)
    return dict(f1=f1, auc=auc, precision=p, recall=r, cm=cm.tolist(), probs=ps, labels=ys)

def save_ckpt(path, model, optimizer, scheduler, scaler, epoch, stage, best_f1):
    torch.save({
        "model": model.state_dict(),
        "opt": optimizer.state_dict() if optimizer else None,
        "sched": scheduler.state_dict() if scheduler else None,
        "scaler": scaler.state_dict() if scaler else None,
        "epoch": epoch, "stage": stage, "best_f1": best_f1,
        "cfg": CFG,
    }, path)

def try_resume(model):
    if os.path.exists(CKPT_LATEST):
        ck = torch.load(CKPT_LATEST, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
        print(f"Resumed: stage={ck['stage']} epoch={ck['epoch']} best_f1={ck['best_f1']:.4f}")
        return ck
    return None

def train_stage(model, loaders, stage, epochs, lr, params, resume_state=None):
    tr_loader, va_loader, _ = loaders
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG["amp"])
    pos_w = torch.tensor([CFG["pos_weight"]], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    start_epoch = 0
    best_f1 = 0.0
    if resume_state and resume_state["stage"] == stage:
        if resume_state["opt"]:    optimizer.load_state_dict(resume_state["opt"])
        if resume_state["sched"]:  scheduler.load_state_dict(resume_state["sched"])
        if resume_state["scaler"]: scaler.load_state_dict(resume_state["scaler"])
        start_epoch = resume_state["epoch"] + 1
        best_f1     = resume_state["best_f1"]
        print(f"  resuming stage {stage} from epoch {start_epoch}")

    for epoch in range(start_epoch, epochs):
        model.train()
        t0 = time.time(); running = 0.0; n = 0
        for wav, y in tr_loader:
            wav = wav.to(device, non_blocking=True); y = y.to(device)
            mel = wav_to_logmel(wav)
            if stage == 2:
                mel = spec_augment(mel, CFG["spec_freq_mask"], CFG["spec_time_mask"])
            img = mel_to_img(mel)

            if stage == 2 and CFG["mixup_alpha"] > 0:
                img, ya, yb, lam = mixup(img, y, CFG["mixup_alpha"])
                with torch.cuda.amp.autocast(enabled=CFG["amp"]):
                    logits = model(img)
                    ya_s = ya * (1 - CFG["label_smooth"]) + 0.5 * CFG["label_smooth"]
                    yb_s = yb * (1 - CFG["label_smooth"]) + 0.5 * CFG["label_smooth"]
                    loss = lam * criterion(logits, ya_s) + (1 - lam) * criterion(logits, yb_s)
            else:
                with torch.cuda.amp.autocast(enabled=CFG["amp"]):
                    logits = model(img)
                    y_s = y * (1 - CFG["label_smooth"]) + 0.5 * CFG["label_smooth"]
                    loss = criterion(logits, y_s)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in params if p.requires_grad], CFG["grad_clip"])
            scaler.step(optimizer); scaler.update()
            running += loss.item() * wav.size(0); n += wav.size(0)
        scheduler.step()

        val = evaluate(model, va_loader)
        dt = time.time() - t0
        print(f"[stage {stage}] epoch {epoch+1}/{epochs}  loss={running/n:.4f}  "
              f"val_f1={val['f1']:.4f} auc={val['auc']:.4f} P={val['precision']:.3f} R={val['recall']:.3f}  ({dt:.0f}s)")

        save_ckpt(CKPT_LATEST, model, optimizer, scheduler, scaler, epoch, stage, best_f1)
        if val["f1"] > best_f1:
            best_f1 = val["f1"]
            save_ckpt(CKPT_BEST, model, optimizer, scheduler, scaler, epoch, stage, best_f1)
            print(f"  ↑ new best val F1 {best_f1:.4f} — saved best_ckpt.pt")

    return best_f1

In [ ]:
model = ASTLite().to(device)
loaders = make_loaders()
resume = try_resume(model)

# ---- Stage 1: head only ----
if resume is None or resume["stage"] <= 1:
    freeze_all_but_head(model)
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"Stage 1 trainable params: {sum(p.numel() for p in params):,}")
    train_stage(model, loaders, stage=1, epochs=CFG["stage1_epochs"],
                lr=CFG["lr_head"], params=params,
                resume_state=resume if (resume and resume['stage']==1) else None)

# ---- Stage 2: unfreeze last 6 blocks ----
unfreeze_last_n_blocks(model, n=6)
params = [p for p in model.parameters() if p.requires_grad]
print(f"Stage 2 trainable params: {sum(p.numel() for p in params):,}")
resume2 = try_resume(model) if (resume and resume['stage']==2) else None
train_stage(model, loaders, stage=2, epochs=CFG["stage2_epochs"],
            lr=CFG["lr_backbone"], params=params, resume_state=resume2)

# ---- Test ----
ck = torch.load(CKPT_BEST, map_location=device, weights_only=False)
model.load_state_dict(ck["model"])
test_metrics = evaluate(model, loaders[2])
print("\n=== TEST ===")
print(f"F1={test_metrics['f1']:.4f}  AUC={test_metrics['auc']:.4f}  "
      f"P={test_metrics['precision']:.3f}  R={test_metrics['recall']:.3f}")
print("Confusion matrix [TN FP; FN TP]:", test_metrics["cm"])
np.save("/kaggle/working/test_probs.npy", test_metrics["probs"])
np.save("/kaggle/working/test_labels.npy", test_metrics["labels"])

In [ ]:
# Reload best checkpoint
ck = torch.load(CKPT_BEST, map_location=device, weights_only=False)
model.load_state_dict(ck["model"])
model.eval()
print(f"Loaded best_ckpt from stage={ck['stage']} epoch={ck['epoch']} val_f1={ck['best_f1']:.4f}")

# ---- 1. Get val probs (single-crop) for threshold search ----
val_eval = evaluate(model, loaders[1])
val_probs, val_labels = val_eval["probs"], val_eval["labels"]

# Sweep thresholds
thresholds = np.linspace(0.30, 0.80, 51)
results = []
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels, preds)
    p, r, _, _ = precision_recall_fscore_support(val_labels, preds, average="binary", zero_division=0)
    results.append((t, f1, p, r))
results = np.array(results)
best_idx = results[:,1].argmax()
best_t, best_f1, best_p, best_r = results[best_idx]
print(f"\nBest val threshold = {best_t:.3f}  F1={best_f1:.4f}  P={best_p:.3f}  R={best_r:.3f}")
print(f"(default 0.5: F1={val_eval['f1']:.4f}  P={val_eval['precision']:.3f}  R={val_eval['recall']:.3f})")

# Show top-5 thresholds for context
top5 = results[results[:,1].argsort()[::-1][:5]]
print("\nTop 5 thresholds (t, F1, P, R):")
for row in top5:
    print(f"  t={row[0]:.3f}  F1={row[1]:.4f}  P={row[2]:.3f}  R={row[3]:.3f}")

In [ ]:
@torch.no_grad()
def evaluate_tta(model, df, n_crops=5):
    """5 crops per clip: 4 corners + center, averaged."""
    model.eval()
    ds = KitchenDS(df, train=False)
    all_probs, all_labels = [], []
    CLIP_LEN_LOCAL = int(CFG["sr"] * CFG["clip_sec"])

    def load_full(path):
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        if sr != CFG["sr"]:
            wav = torchaudio.functional.resample(wav, sr, CFG["sr"])
        return wav.squeeze(0)

    for i in range(len(df)):
        row = df.iloc[i]
        wav = load_full(row["abs_path"])
        if wav.numel() < CLIP_LEN_LOCAL:
            wav = F.pad(wav, (0, CLIP_LEN_LOCAL - wav.numel()))
        L = wav.numel()
        if L == CLIP_LEN_LOCAL:
            crops = [wav]
        else:
            positions = np.linspace(0, L - CLIP_LEN_LOCAL, n_crops).astype(int)
            crops = [wav[s:s+CLIP_LEN_LOCAL] for s in positions]
        batch = torch.stack(crops).to(device)
        mel = wav_to_logmel(batch)
        img = mel_to_img(mel)
        with torch.amp.autocast("cuda", enabled=CFG["amp"]):
            logits = model(img)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs.mean())
        all_labels.append(float(row["label"]))
    return np.array(all_probs), np.array(all_labels)

# TTA on val to re-tune threshold with TTA probs
print("Running TTA on val...")
val_probs_tta, val_labels_tta = evaluate_tta(model, val_df, n_crops=5)
results_tta = []
for t in thresholds:
    preds = (val_probs_tta >= t).astype(int)
    f1 = f1_score(val_labels_tta, preds)
    p, r, _, _ = precision_recall_fscore_support(val_labels_tta, preds, average="binary", zero_division=0)
    results_tta.append((t, f1, p, r))
results_tta = np.array(results_tta)
best_t_tta = results_tta[results_tta[:,1].argmax(), 0]
print(f"Best val threshold w/ TTA = {best_t_tta:.3f}  F1={results_tta[results_tta[:,1].argmax(),1]:.4f}")

# TTA on test, apply tuned threshold
print("\nRunning TTA on test...")
test_probs_tta, test_labels_tta = evaluate_tta(model, test_df, n_crops=5)

for label, t in [("default 0.5", 0.5), ("val-tuned (single)", best_t), ("val-tuned (TTA)", best_t_tta)]:
    preds = (test_probs_tta >= t).astype(int)
    f1 = f1_score(test_labels_tta, preds)
    p, r, _, _ = precision_recall_fscore_support(test_labels_tta, preds, average="binary", zero_division=0)
    auc = roc_auc_score(test_labels_tta, test_probs_tta)
    cm = confusion_matrix(test_labels_tta, preds)
    print(f"\n[{label}, t={t:.3f}]  F1={f1:.4f}  AUC={auc:.4f}  P={p:.3f}  R={r:.3f}")
    print(f"  CM [TN FP; FN TP]: {cm.tolist()}")

# Save TTA probs for ensembling later
np.save("/kaggle/working/test_probs_tta_v6.npy", test_probs_tta)
np.save("/kaggle/working/test_labels_v6.npy", test_labels_tta)
np.save("/kaggle/working/val_probs_tta_v6.npy", val_probs_tta)
np.save("/kaggle/working/val_labels_v6.npy", val_labels_tta)
print("\nSaved TTA probs to /kaggle/working/ for ensembling")

In [ ]:
import IPython.display as ipd
import pandas as pd

# Recompute with the winning threshold
THRESH = 0.610
preds_tta = (test_probs_tta >= THRESH).astype(int)

audit = test_df.copy().reset_index(drop=True)
audit["true"] = test_labels_tta.astype(int)
audit["prob"] = test_probs_tta
audit["pred"] = preds_tta
audit["error_type"] = "correct"
audit.loc[(audit["true"]==0) & (audit["pred"]==1), "error_type"] = "FP"  # noncooking → cooking
audit.loc[(audit["true"]==1) & (audit["pred"]==0), "error_type"] = "FN"  # cooking → noncooking
audit["confidence_margin"] = (audit["prob"] - THRESH).abs()

fps = audit[audit.error_type=="FP"].sort_values("prob", ascending=False)  # most confident wrong
fns = audit[audit.error_type=="FN"].sort_values("prob", ascending=True)   # most confident wrong

print(f"FPs: {len(fps)}  |  FNs: {len(fns)}")
print("\n=== TOP 15 MOST CONFIDENT FALSE POSITIVES (model said cooking, label says noncooking) ===")
cols = ["abs_path", "prob", "confidence_margin"]
print(fps[cols].head(15).to_string())

print("\n=== TOP 15 MOST CONFIDENT FALSE NEGATIVES (model said noncooking, label says cooking) ===")
print(fns[cols].head(15).to_string())

# Save full audit
audit.to_csv("/kaggle/working/v6_error_audit.csv", index=False)
print("\nSaved /kaggle/working/v6_error_audit.csv")

In [ ]:
# Run this cell, listen to each clip, mark in a notebook: "mislabeled" / "genuinely hard" / "ambiguous"
print("=== FALSE POSITIVES (label=noncooking, model confident it's cooking) ===")
for _, row in fps.head(10).iterrows():
    print(f"\nprob={row['prob']:.3f}  |  {row['abs_path']}")
    display(ipd.Audio(row['abs_path']))

print("\n=== FALSE NEGATIVES (label=cooking, model confident it's noncooking) ===")
for _, row in fns.head(10).iterrows():
    print(f"\nprob={row['prob']:.3f}  |  {row['abs_path']}")
    display(ipd.Audio(row['abs_path']))

In [ ]:
# Cell 12 — Export inference bundle
import json, shutil

INFER_CFG = {
    "sr": CFG["sr"],
    "clip_sec": CFG["clip_sec"],
    "n_mels": CFG["n_mels"],
    "n_fft": CFG["n_fft"],
    "hop": CFG["hop"],
    "fmin": CFG["fmin"],
    "fmax": CFG["fmax"],
    "img_size": CFG["img_size"],
    "threshold": 0.610,
    "tta_crops": 5,
    "model_arch": "deit_small_patch16_224",
    "feature_dim": 384,
    "version": "v6_ast_only",
    "val_f1_tta": 0.8980,
    "test_f1_tta": 0.8958,
    "test_auc": 0.9545,
}
with open("/kaggle/working/infer_config.json", "w") as f:
    json.dump(INFER_CFG, f, indent=2)
print("Saved infer_config.json")
print(json.dumps(INFER_CFG, indent=2))